# Naive Bayes

Naive Bayes flips the question around: instead of drawing a boundary, it models
*each class's distribution* and asks "which class most likely produced this
point?" (Bayes' rule). The **"naive"** part is assuming features are independent
given the class — usually false, yet the classifier is often a **fast, surprisingly
strong baseline**, and training is essentially just computing per-class means and
variances.

For continuous features (like breast-cancer measurements) the right variant is
**Gaussian** Naive Bayes: each feature is modelled as a per-class bell curve. We
use `smartcore`'s `GaussianNB` on the same dataset as the other
[classification](knn-classification.ipynb) models, so the closing comparison is
apples-to-apples.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::dataset::breast_cancer;
use smartcore::model_selection::train_test_split;
use smartcore::metrics::{accuracy, precision, recall, f1};
use smartcore::naive_bayes::gaussian::{GaussianNB, GaussianNBParameters};
use smartcore::api::SupervisedEstimator;
use plotters::prelude::*;

let (data, y, p): (Vec<f32>, Vec<u32>, usize) = {
    let ds = breast_cancer::load_dataset();
    (ds.data.clone(), ds.target.iter().map(|&v| v as u32).collect(), ds.num_features)
};
let x: DenseMatrix<f32> = DenseMatrix::new(y.len(), p, data.clone(), false);

// Fit + evaluate. GaussianNB's inherent fit takes optional class priors
// (None = estimate them from the data).
let (yte, pred): (Vec<u32>, Vec<u32>) = {
    let (xtr, xte, ytr, yte) = train_test_split(&x, &y, 0.3, true, Some(42));
    let model = GaussianNB::fit(&xtr, &ytr, GaussianNBParameters { priors: None }).unwrap();
    (yte, model.predict(&xte).unwrap())
};
let yte_f: Vec<f32> = yte.iter().map(|&v| v as f32).collect();
let pred_f: Vec<f32> = pred.iter().map(|&v| v as f32).collect();
println!("GaussianNB on breast cancer:");
println!("  accuracy  = {:.3}", accuracy(&yte, &pred));
println!("  precision = {:.3}", precision(&yte_f, &pred_f));
println!("  recall    = {:.3}", recall(&yte_f, &pred_f));
println!("  f1        = {:.3}", f1(&yte_f, &pred_f, 1.0));

## What the model actually stores

GaussianNB's entire "training" is: for each class, the **mean and variance of
every feature**. At prediction time it plugs a new point into those Gaussians and
multiplies (the naive independence assumption). Here are the two per-class bell
curves it fits for one feature (feature 0, "mean radius") — the separation between
them is exactly what the model exploits:

In [ ]:
{  // block-scoped so the `pdf` closure is never persisted across cells
// Per-class mean/std of feature 0, computed by hand to show what GaussianNB stores.
let (m0, s0, m1, s1): (f64, f64, f64, f64) = {
    let mut v0: Vec<f64> = vec![];
    let mut v1: Vec<f64> = vec![];
    for i in 0..y.len() { if y[i] == 0 { v0.push(data[i * p] as f64); } else { v1.push(data[i * p] as f64); } }
    // `stat` captures nothing (data comes via the argument), so it's safe to define here.
    let stat = |v: &[f64]| {
        let m = v.iter().sum::<f64>() / v.len() as f64;
        let sd = (v.iter().map(|x| (x - m).powi(2)).sum::<f64>() / v.len() as f64).sqrt();
        (m, sd)
    };
    let (m0, s0) = stat(&v0);
    let (m1, s1) = stat(&v1);
    (m0, s0, m1, s1)
};
println!("class 0 (malignant): mean={:.1}, sd={:.1}", m0, s0);
println!("class 1 (benign):    mean={:.1}, sd={:.1}", m1, s1);

let lo = (m0.min(m1) - 3.0 * s0.max(s1)).max(0.0);
let hi = m0.max(m1) + 3.0 * s0.max(s1);
let pdf = |x: f64, m: f64, s: f64| (-(x - m).powi(2) / (2.0 * s * s)).exp() / (s * (2.0 * std::f64::consts::PI).sqrt());
let curve0: Vec<(f64, f64)> = (0..200).map(|i| { let x = lo + (hi - lo) * i as f64 / 199.0; (x, pdf(x, m0, s0)) }).collect();
let curve1: Vec<(f64, f64)> = (0..200).map(|i| { let x = lo + (hi - lo) * i as f64 / 199.0; (x, pdf(x, m1, s1)) }).collect();
let ymax = curve0.iter().chain(curve1.iter()).map(|(_, y)| *y).fold(0.0, f64::max) * 1.1;

evcxr_figure((580, 360), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("per-class Gaussians (feature 0)", ("sans-serif", 16))
        .margin(10).x_label_area_size(34).y_label_area_size(44)
        .build_cartesian_2d(lo..hi, 0f64..ymax)?;
    chart.configure_mesh().x_desc("mean radius").y_desc("density").draw()?;
    chart.draw_series(LineSeries::new(curve0.clone(), RGBColor(220, 120, 20).stroke_width(2)))?
        .label("malignant").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RGBColor(220, 120, 20)));
    chart.draw_series(LineSeries::new(curve1.clone(), RGBColor(30, 90, 200).stroke_width(2)))?
        .label("benign").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RGBColor(30, 90, 200)));
    chart.configure_series_labels().position(SeriesLabelPosition::UpperRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})
}

The two curves overlap but their centres are well apart — a single feature
already carries real signal, and GaussianNB combines all 30 this way.

## Strengths, and the naive caveat

- **Fast & cheap** — training is one pass to compute means/variances; no iterative
  optimization. A strong first baseline to beat before reaching for heavier models.
- **The independence assumption is wrong** — breast-cancer features are highly
  correlated (radius, perimeter, area all measure size). NB ignores that, which is
  why it's usually beaten by models that don't (see the comparison in the
  [SVM chapter](svm-classification.ipynb)) — but it's remarkably competitive for
  how little it costs.

`smartcore` also ships `MultinomialNB`/`BernoulliNB` (count/binary features) and
`CategoricalNB` — pick the variant matching your feature types.

Next: [SVM](svm-classification.ipynb), and a head-to-head of all four classifiers.